In [74]:
import pandas as pd
import numpy as np

tickers = ["MSI","MO","MCD","PG","SO","GE","TRV","DG","USFD","RRX","DVA"]

market_values_eur = {
    "MSI": 5127202.18,
    "MO": 5763912.65,

    "MCD": 6177011.08,
    "PG": 6316127.33,
    "SO": 6103066.08,
    "GE": 6282228.64,
    "TRV": 5207148.28,
    "DG": 4000000.75,
    "USFD": 4767310.01,
    "RRX": 5000000.00,
    "DVA": 5000000.00,
}

h = pd.DataFrame({
    "ticker": tickers,
    "market_value_eur": [market_values_eur[t] for t in tickers]
})

h["weight"] = h["market_value_eur"] / h["market_value_eur"].sum()
h = h.sort_values("weight", ascending=False).reset_index(drop=True)

print("Total NAV (EUR):", h["market_value_eur"].sum())
display(h.assign(weight_pct=(h["weight"]*100)).round({"market_value_eur":2, "weight":6, "weight_pct":2}))

Total NAV (EUR): 59744006.99999999


,ticker,market_value_eur,weight,weight_pct
0,PG,6316127.33,0.105720,10.57
1,GE,6282228.64,0.105152,10.52
2,MCD,6177011.08,0.103391,10.34
3,SO,6103066.08,0.102154,10.22
4,MO,5763912.65,0.096477,9.65
5,TRV,5207148.28,0.087158,8.72
6,MSI,5127202.18,0.085820,8.58
7,RRX,5000000.00,0.083690,8.37
8,DVA,5000000.00,0.083690,8.37
9,USFD,4767310.01,0.079796,7.98


In [75]:
# holdings tickers (giữ nguyên để bạn đọc dễ)
tickers = ["MSI","MO","MCD","PG","SO","GE","TRV","DG","USFD","RRX","DVA"]

# map sang ticker đúng trên yfinance
ticker_map = {
    "DG": "DG.PA",       # Vinci - Euronext Paris
    "APAM": "APAM.AS",   # APAM - Euronext Amsterdam
    # các mã US giữ nguyên
}

yf_tickers = [ticker_map.get(t, t) for t in tickers]
print("Tickers for yfinance:", yf_tickers)

Tickers for yfinance: ['MSI', 'MO', 'MCD', 'PG', 'SO', 'GE', 'TRV', 'DG.PA', 'USFD', 'RRX', 'DVA']


In [76]:
def portfolio_healthcheck(h):
    n = len(h)
    top1 = float(h["weight"].iloc[0])
    top5 = float(h["weight"].head(5).sum())
    top10 = float(h["weight"].head(10).sum())
    small = int((h["weight"] < 0.005).sum())   # <0.5%
    tiny  = int((h["weight"] < 0.0025).sum())  # <0.25%
    return pd.DataFrame([{
        "n_positions": n,
        "top1_weight": top1,
        "top5_weight_sum": top5,
        "top10_weight_sum": top10,
        "count_<0.5%": small,
        "count_<0.25%": tiny,
        "weight_sum_check": float(h["weight"].sum())
    }])

display(portfolio_healthcheck(h))

,n_positions,top1_weight,top5_weight_sum,top10_weight_sum,count_<0.5%,count_<0.25%,weight_sum_check
0,11,0.10572,0.512894,0.933048,0,0,1.0


In [77]:
# pip install yfinance --quiet
import yfinance as yf

def download_prices(tickers, period="9mo"):
    px = yf.download(tickers, period=period, interval="1d", auto_adjust=True, progress=False)
    # yfinance trả MultiIndex: ("Close", ticker)
    close = px["Close"].copy()
    close = close.dropna(how="all")
    return close

px_usd = download_prices(tickers, period="9mo")
display(px_usd.tail())

# check ticker nào thiếu dữ liệu
missing = [t for t in tickers if t not in px_usd.columns or px_usd[t].dropna().empty]
print("Missing tickers:", missing)

Ticker,DG,DVA,GE,MCD,MO,MSI,PG,RRX,SO,TRV,USFD
Date,,,,,,,,,,,
2026-02-19,151.789993,147.339996,334.739990,327.109985,67.989998,453.679993,158.559998,213.710007,95.050003,299.899994,96.379997
2026-02-20,150.639999,150.729996,343.220001,329.230011,67.570000,462.760010,160.779999,215.360001,94.300003,304.929993,96.709999
2026-02-23,152.899994,151.279999,338.989990,334.559998,68.980003,465.029999,165.169998,218.210007,95.180000,305.390015,96.129997
2026-02-24,153.960007,150.910004,345.640015,333.049988,69.250000,470.850006,165.279999,223.690002,95.809998,305.429993,96.550003
2026-02-25,154.229996,148.875000,343.820007,332.500000,69.480003,469.329987,163.660004,221.565002,95.474998,304.709991,95.400002


Missing tickers: []


In [78]:
fx = download_prices(["EURUSD=X"], period="9mo")  # 1 EUR = ? USD
eurusd = fx["EURUSD=X"].dropna()

# USD -> EUR price: price_usd / (USD per EUR)
# (vì 1 EUR = eurusd USD => 1 USD = 1/eurusd EUR)
px_eur = px_usd.div(eurusd, axis=0)

display(px_eur.tail())

Ticker,DG,DVA,GE,MCD,MO,MSI,PG,RRX,SO,TRV,USFD
Date,,,,,,,,,,,
2026-02-19,128.757371,124.982617,283.946525,277.474298,57.673192,384.838565,134.500095,181.281638,80.627110,254.393152,81.755291
2026-02-20,127.995799,128.072267,291.627181,279.740166,57.412879,393.197939,136.611554,182.987092,80.124829,259.092926,82.172555
2026-02-23,129.195911,127.827063,286.436378,282.693167,58.286035,392.936406,139.563696,184.380913,80.424247,258.045406,81.226965
2026-02-24,130.542691,127.956593,293.068171,282.393087,58.717076,399.233724,140.140912,189.666755,81.237298,258.974093,81.864748
2026-02-25,130.601969,126.067359,291.146802,281.561019,58.835671,397.428660,138.587301,187.621257,80.848234,258.028439,80.784727


In [79]:
def compute_metrics(px: pd.DataFrame, h: pd.DataFrame):
    tickers = h["ticker"].tolist()
    px = px[tickers].dropna(how="all")
    rets = px.pct_change().dropna(how="all")

    # momentum (trading days): 21~1M, 42~2M
    m30 = (px.iloc[-1] / px.iloc[-21] - 1) if len(px) >= 21 else pd.Series(index=tickers, dtype=float)
    m60 = (px.iloc[-1] / px.iloc[-42] - 1) if len(px) >= 42 else pd.Series(index=tickers, dtype=float)
    m30.name, m60.name = "mom_30d", "mom_60d"

    # vol 30d annualized
    vol30 = (rets.tail(21).std() * np.sqrt(252)).rename("vol_30d_ann")

    # max drawdown last 90d (or available)
    w = min(90, len(px))
    wpx = px.tail(w)
    wealth = (1 + wpx.pct_change().fillna(0)).cumprod()
    peak = wealth.cummax()
    dd = (wealth/peak - 1).min().rename(f"max_dd_{w}d")

    # risk contribution (cov 63d)
    cov = rets.tail(63).cov() * 252
    wgt = h.set_index("ticker")["weight"].reindex(cov.index).fillna(0.0)

    port_var = float(wgt.T @ cov.values @ wgt.values)
    if port_var > 0:
        mrc = cov.values @ wgt.values
        rc = (wgt.values * mrc) / port_var
        rc = pd.Series(rc, index=cov.index, name="risk_contrib_share")
    else:
        rc = pd.Series(index=cov.index, dtype=float, name="risk_contrib_share")

    out = pd.DataFrame(index=cov.index)
    out = out.join(h.set_index("ticker")[["market_value_eur","weight"]], how="left")
    out = out.join(m30, how="left").join(m60, how="left")
    out = out.join(vol30, how="left").join(dd, how="left")
    out = out.join(rc, how="left")
    out = out.reset_index(names="ticker")
    return out.sort_values("weight", ascending=False)

metrics = compute_metrics(px_eur, h)   # hoặc px_usd
display(metrics.round({
    "market_value_eur":2, "weight":6, "mom_30d":4, "mom_60d":4, "vol_30d_ann":4, "risk_contrib_share":4
}))

,ticker,market_value_eur,weight,mom_30d,mom_60d,vol_30d_ann,max_dd_90d,risk_contrib_share
0,PG,6316127.33,0.105720,0.1096,0.1394,0.2067,-0.100198,0.0701
1,GE,6282228.64,0.105152,0.1624,0.0854,0.3173,-0.132275,0.0843
2,MCD,6177011.08,0.103391,0.0645,0.0600,0.1724,-0.058893,0.0608
3,SO,6103066.08,0.102154,0.0893,0.1026,0.2635,-0.141934,0.0759
4,MO,5763912.65,0.096477,0.0983,0.1982,0.3155,-0.143548,0.0694
5,TRV,5207148.28,0.087158,0.0895,0.0358,0.2057,-0.087819,0.0461
6,MSI,5127202.18,0.085820,0.1623,0.2452,0.3115,-0.193815,0.0761
7,RRX,5000000.00,0.083690,0.4414,0.5289,0.5527,-0.136597,0.1218
8,DVA,5000000.00,0.083690,0.4177,0.2997,0.8424,-0.208957,0.2032
9,USFD,4767310.01,0.079796,0.1467,0.2477,0.5444,-0.083284,0.1325


In [80]:
def flag_trim_cut(metrics,
                  w_big=0.09,      # >=9% coi là big
                  w_tiny=0.01,     # <=1% coi là bụi (vì bạn chỉ có 12 mã nên đặt 1% hợp lý)
                  mom_bad=0.0,
                  rc_high=0.14,    # >14% share risk là cao (12 mã thì 8.3% là đều)
                  dd_bad=-0.18):   # drawdown dưới -18% thì đáng chú ý
    m = metrics.copy()

    dd_col = [c for c in m.columns if c.startswith("max_dd_")]
    dd_series = m[dd_col[0]] if dd_col else pd.Series([np.nan]*len(m), index=m.index)

    m["flag_big_neg_mom30"] = (m["weight"] >= w_big) & (m["mom_30d"] < mom_bad)
    m["flag_high_risk_contrib"] = (m["risk_contrib_share"] >= rc_high)
    m["flag_deep_drawdown"] = (dd_series <= dd_bad)
    m["flag_tiny_weight"] = (m["weight"] <= w_tiny)

    m["trim_score"] = (
        m["flag_big_neg_mom30"].astype(int) * 3
        + m["flag_high_risk_contrib"].astype(int) * 2
        + m["flag_deep_drawdown"].astype(int) * 1
        + m["flag_tiny_weight"].astype(int) * 1
    )

    cols = ["ticker","market_value_eur","weight","mom_30d","mom_60d","vol_30d_ann"] + dd_col + ["risk_contrib_share",
            "flag_big_neg_mom30","flag_high_risk_contrib","flag_deep_drawdown","flag_tiny_weight","trim_score"]
    return m.sort_values(["trim_score","weight"], ascending=[False, False])[cols]

flags = flag_trim_cut(metrics)
display(flags.round({
    "market_value_eur":2, "weight":6, "mom_30d":4, "mom_60d":4, "vol_30d_ann":4, "risk_contrib_share":4
}))

,ticker,market_value_eur,weight,mom_30d,mom_60d,vol_30d_ann,max_dd_90d,risk_contrib_share,flag_big_neg_mom30,flag_high_risk_contrib,flag_deep_drawdown,flag_tiny_weight,trim_score
8,DVA,5000000.00,0.083690,0.4177,0.2997,0.8424,-0.208957,0.2032,False,True,True,False,3
6,MSI,5127202.18,0.085820,0.1623,0.2452,0.3115,-0.193815,0.0761,False,False,True,False,1
0,PG,6316127.33,0.105720,0.1096,0.1394,0.2067,-0.100198,0.0701,False,False,False,False,0
1,GE,6282228.64,0.105152,0.1624,0.0854,0.3173,-0.132275,0.0843,False,False,False,False,0
2,MCD,6177011.08,0.103391,0.0645,0.0600,0.1724,-0.058893,0.0608,False,False,False,False,0
3,SO,6103066.08,0.102154,0.0893,0.1026,0.2635,-0.141934,0.0759,False,False,False,False,0
4,MO,5763912.65,0.096477,0.0983,0.1982,0.3155,-0.143548,0.0694,False,False,False,False,0
5,TRV,5207148.28,0.087158,0.0895,0.0358,0.2057,-0.087819,0.0461,False,False,False,False,0
7,RRX,5000000.00,0.083690,0.4414,0.5289,0.5527,-0.136597,0.1218,False,False,False,False,0
9,USFD,4767310.01,0.079796,0.1467,0.2477,0.5444,-0.083284,0.1325,False,False,False,False,0


In [81]:
import numpy as np
import pandas as pd

def portfolio_returns(px: pd.DataFrame, weights: pd.Series):
    rets = px.pct_change().dropna(how="all")
    weights = weights.reindex(rets.columns).fillna(0.0)
    # daily portfolio return
    port_ret = rets.mul(weights, axis=1).sum(axis=1)
    return port_ret, rets

w = h.set_index("ticker")["weight"]
port_ret, asset_rets = portfolio_returns(px_eur, w)  # hoặc px
port_ret.tail()

/var/folders/d3/4j2xrjxn06v329lnxwrwmzs80000gn/T/ipykernel_31757/852297339.py:5: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = px.pct_change().dropna(how="all")


Date
2026-02-19    0.012682
2026-02-20    0.010618
2026-02-23    0.003131
2026-02-24    0.010006
2026-02-25   -0.006327
dtype: float64

In [82]:
def perf_stats(r: pd.Series, rf_annual=0.0):
    r = r.dropna()
    if len(r) < 5:
        raise ValueError("Không đủ dữ liệu returns.")

    ann_factor = 252
    rf_daily = (1 + rf_annual)**(1/ann_factor) - 1

    excess = r - rf_daily

    ann_return = (1 + r).prod()**(ann_factor/len(r)) - 1
    ann_vol = r.std() * np.sqrt(ann_factor)

    sharpe = np.nan
    if ann_vol > 0:
        sharpe = (excess.mean() * ann_factor) / (r.std() * np.sqrt(ann_factor))

    downside = r[r < 0]
    downside_dev = downside.std() * np.sqrt(ann_factor) if len(downside) > 1 else np.nan
    sortino = (excess.mean() * ann_factor) / downside_dev if downside_dev and downside_dev > 0 else np.nan

    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    max_dd = dd.min()

    calmar = ann_return / abs(max_dd) if max_dd < 0 else np.nan

    return pd.Series({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar
    })

stats = perf_stats(port_ret, rf_annual=0.0)
stats

ann_return      0.241934
ann_vol         0.124215
sharpe          1.806612
sortino         3.380835
max_drawdown   -0.067988
calmar          3.558470
dtype: float64

In [83]:
def efficiency_stats(r: pd.Series):
    r = r.dropna()
    hit_rate = (r > 0).mean()
    avg_gain = r[r > 0].mean()
    avg_loss = r[r < 0].mean()
    payoff = (avg_gain / abs(avg_loss)) if (avg_loss < 0) else np.nan
    return pd.Series({
        "hit_rate": hit_rate,
        "avg_gain": avg_gain,
        "avg_loss": avg_loss,
        "payoff_ratio": payoff
    })

eff = efficiency_stats(port_ret)
eff

hit_rate        0.541237
avg_gain        0.006320
avg_loss       -0.005913
payoff_ratio    1.068716
dtype: float64

In [84]:
import yfinance as yf

bench = yf.download(["SPY"], period="9mo", interval="1d", auto_adjust=True, progress=False)["Close"]["SPY"]
bench_ret = bench.pct_change().dropna()

# align dates
df = pd.concat([port_ret.rename("port"), bench_ret.rename("bench")], axis=1).dropna()
active = df["port"] - df["bench"]

tracking_error = active.std() * np.sqrt(252)
info_ratio = (active.mean() * 252) / tracking_error if tracking_error > 0 else np.nan

pd.Series({"tracking_error_ann": tracking_error, "information_ratio": info_ratio})

tracking_error_ann    0.151959
information_ratio     0.006386
dtype: float64

In [85]:
import pandas as pd

df = pd.concat([port_ret.rename("port"), bench_ret.rename("spy")], axis=1).dropna()
cum = (1+df).cumprod()

total_port = cum["port"].iloc[-1] - 1
total_spy  = cum["spy"].iloc[-1] - 1
print("Total return sample: port =", round(total_port*100,2), "% | SPY =", round(total_spy*100,2), "%")

Total return sample: port = 18.15 % | SPY = 18.24 %


In [86]:
import numpy as np
import pandas as pd

def suggest_two_sells_us(px_eur_or_usd, h, metrics, bench_ret):
    # daily returns
    asset_ret = px_eur_or_usd.pct_change().dropna(how="all")
    w = h.set_index("ticker")["weight"].reindex(asset_ret.columns).fillna(0.0)

    # align benchmark
    bench_ret = bench_ret.reindex(asset_ret.index).dropna()
    asset_ret = asset_ret.loc[bench_ret.index]

    # approx active contribution: weight * (asset - benchmark)
    active_contrib = (asset_ret.sub(bench_ret, axis=0).mul(w, axis=1)).mean() * 252
    active_contrib = active_contrib.rename("active_contrib_ann")

    m = metrics.set_index("ticker").copy().join(active_contrib, how="left")

    # Score 1: risk offender (ưu tiên risk_contrib_share + vol)
    m["risk_offender"] = 0.7*m["risk_contrib_share"].fillna(0) + 0.3*m["vol_30d_ann"].fillna(0)

    # Score 2: alpha offender (active contrib càng âm càng tệ, thêm phạt mom âm)
    m["alpha_offender"] = (-m["active_contrib_ann"].fillna(0)) + 0.3*(m["mom_30d"] < 0).astype(int)

    sell1 = m.sort_values("risk_offender", ascending=False).head(1)
    m2 = m.drop(index=sell1.index)
    sell2 = m2.sort_values("alpha_offender", ascending=False).head(1)

    sells = pd.concat([sell1, sell2])
    cols = ["weight","mom_30d","vol_30d_ann","max_dd_90d","risk_contrib_share","active_contrib_ann","risk_offender","alpha_offender"]
    return sells[cols].sort_values("risk_offender", ascending=False)

sells_today = suggest_two_sells_us(px_eur, h, metrics, bench_ret)  # hoặc px_usd
display(sells_today)

/var/folders/d3/4j2xrjxn06v329lnxwrwmzs80000gn/T/ipykernel_31757/380080807.py:6: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  asset_ret = px_eur_or_usd.pct_change().dropna(how="all")


,weight,mom_30d,vol_30d_ann,max_dd_90d,risk_contrib_share,active_contrib_ann,risk_offender,alpha_offender
ticker,,,,,,,,
DVA,0.08369,0.417704,0.842378,-0.208957,0.203157,-0.010354,0.394923,0.010354
PG,0.10572,0.109566,0.206677,-0.100198,0.070072,-0.028497,0.111053,0.028497
